# TAP access to CSA metadata

To install [astroquery](https://astroquery.readthedocs.io/en/latest/index.html) ([Ginsburg et al, 2019](https://doi.org/10.3847/1538-3881/aafc33)) to be able to use [TAP+](https://astroquery.readthedocs.io/en/latest/utils/tap.html):

`conda install -c astropy astroquery`

The different missions have their own TAP servers; Cluster's is:
`https://csa.esac.esa.int/csa-sl-tap/tap/`

## Tables: metadata of metadata

First of all, it's helpful to know what tables are there. 

From example 1.1 on [Astroquery TAP page](https://astroquery.readthedocs.io/en/latest/utils/tap.html), load the table objects using the method `load_tables()`, then print the names of those table objects using the `get_qualified_name()` method. This returns the schema (e.g., csa.) and the table names (e.g., csa.dataset). We only need to focus on the `csa.` schema tables.

In [1]:
from astroquery.utils.tap.core import TapPlus

CSA = TapPlus(url="https://csa.esac.esa.int/csa-sl-tap/tap/")
tables = CSA.load_tables()#only_names=True)

for table in tables:
    print(table.get_qualified_name())

Created TAP+ (v20200428.1) - Connection:
	Host: csa.esac.esa.int
	Use HTTPS: True
	Port: 443
	SSL Port: 443
INFO: Retrieving tables... [astroquery.utils.tap.core]
INFO: Parsing tables... [astroquery.utils.tap.core]
INFO: Done. [astroquery.utils.tap.core]
csa.csa.array_dimension
csa.csa.cluster_data_type
csa.csa.dataset
csa.csa.dataset_contact
csa.csa.dataset_inventory
csa.csa.dataset_referenced
csa.csa.experiment
csa.csa.experiment_dataset
csa.csa.file
csa.csa.instrument_type
csa.csa.measurement_type
csa.csa.news
csa.csa.observatory
csa.csa.parameter
csa.csa.parameter_data
csa.csa.parameter_dependency
csa.csa.parameter_label
csa.csa.parameter_representation
csa.csa.pregen_inventory
csa.csa.processing_level
csa.csa.v_dataset
csa.csa.v_dataset_inventory
csa.csa.v_dataset_referenced
csa.csa.v_experiment
csa.csa.v_file
csa.csa.v_instrument
csa.csa.v_mission
csa.csa.v_observatory
csa.csa.v_parameter
csa.csa.v_pregen_inventory
csa.csa.v_quick_look_dataset
graphics.graphics.distribution_panel

From the above list of tables, the helpful ones are those beginning `csa.csa.` and the ones meant for users begin with `csa.csa.v_`, where the 'v' means 'view'. The view tables contain the same information as the other `csa.csa.` tables but the columns in those tables might be grouped slightly differently, and the columns containing information meant for internal database use have been taken out since they are of no use to the end-user.

The schema (the first `csa.`) isn't needed when making the query.

Also from [Astroquery TAP page](https://astroquery.readthedocs.io/en/latest/utils/tap.html), load a specific table and print out the column names. 

In [2]:
table = CSA.load_table('csa.v_parameter')

print('\n Table: \n', table, '\n')

print('Column names: ')
for column in (table.columns):
    print(column.name)

Retrieving table 'csa.v_parameter'

 Table: 
 TAP Table name: csa.csa.v_parameter
Description: 
Num. columns: 37 

Column names: 
array_dimension
cat_description
caveat
compound
compound_definition
coordinate_system
dataset_id
data_type
delta_minus
delta_plus
display_type
entity
error_minus
error_plus
field_name
fillval
fluctuations
frame
frame_origin
frame_velocity
label_axis
parameter_data
parameter_dependency
parameter_id
parameter_label
parameter_representation
property
quality
scale_max
scale_min
scale_type
si_conversion
significant_digits
target_system
tensor_order
units
value_type


## Querying Tables - accessing metadata

So far we have only been loading metadata of the metadata, so we need to make an actual query to get the values:

From [AppDynamics](https://docs.appdynamics.com/display/PRO45/ADQL+Queries) rather than [Astronomical Data Query Language](https://www.ivoa.net/documents/ADQL/2.0), but still usefully put:

`SELECT * | {[DISTINCT] expression [AS alias] [, expression [AS alias]]...}
FROM event_type
[WHERE condition_expression]
[SINCE timevalue [UNTIL timevalue]]
[HAVING condition_expression]
[ORDER BY {field_name | alias} [ASC | DESC] [, {field_name | alias} [ASC | DESC]]...]
[LIMIT integer [, integer]...]`

While the SELECT and FROM clauses are required, all other clauses are optional. Type spaces around each keyword. 

### Whole table

To see all the columns and first 2000 rows (TAP+ has a synchronous request limit of 2000) of a table:

In [3]:
results = CSA.launch_job("SELECT * FROM csa.v_dataset")
r = results.get_results()
r

acknowledgement,allow_overlap,category,compress,crosscal,dataset_caveats,dataset_id,dataset_reference,dataset_version,data_type,data_type_id,date_last_file_ingested,date_last_update,description,display_order,end_date,experiments,file_extension,gui_name,ingestion_date,instruments,interval,inventory,is_cef,is_istp,key_product,level,main_group,max_start_file_date,max_time_resolution,measurement_types,min_time_resolution,multi_inst,multi_sc,observatory_name,on_demand,person_reference,pregen_product,quicklook_product,released,start_date,time_resolution,title,unit_name
object,int16,object,int16,int16,object,object,object,object,object,object,object,object,object,int32,object,object,object,object,object,object,object,int16,bool,bool,int16,object,object,object,float64,object,float64,int16,int16,object,int16,object,int16,int16,int16,object,float64,object,object
Please acknowledge the instrument team and ESA Cluster Active Archive in any publication based upon use of this data.,0,General,0,0,*D1_CQ_STA-DWP_PSD_CAV,D1_CG_STA-DWP_COMBI_PNG,D1_CQ_STA-DWP_PSD_CAV (1),,CAA_Graphic,CG,2018-11-22T19:55:10.971Z,2020-11-10T03:22:28.202Z,"This dataset contains 1 minute resolution Power Spectral Density measurements of the magnetic field in two bands. The lower band contains 33 frequencies distributed linearly between 2Hz and 10Hz, while the upper band contains 27 frequencies distributed logarithmically between 20 Hz and 4 kHz. It is measured along a single axis orientated for minimum interference.",30020,2007-09-30T00:00:00.000Z,STAFF-DWP,png,"Plot (24 hr) - 1D SA/SC Power Spectra Density (2-4000 Hz, 1-min) [png]",2018-01-18T09:41:53.279Z,DWP-D1,24 hour,0,False,False,0,Calibrated,Graphical,2007-09-29T00:00:00.000Z,--,"Magnetic_Field, Radio_and_Plasma_Waves",--,0,0,DoubleStar-1,0,KH Yearby>DWP Technical Manager>K.H.Yearby@sheffield.ac.uk,0,1,1,2004-02-10T00:00:00.000Z,86400.0,Power Spectral Density (1 minute resolution) (png),Bitmap
Please acknowledge the CIS team and ESA Cluster Science Archive when using this data.,0,Particle Distribution,1,0,*C4_CQ_CIS-CODIF_CAVEATS,C4_CP_CIS-CODIF_HS_O1_PEF,C4_CQ_CIS-CODIF_CAVEATS (1),,CAA_Parameter,CP,2021-04-10T22:56:10.556Z,2021-04-11T02:14:43.366Z,"This dataset contains CIS-CODIF 3D Oxygen+ distributions for the High-Sensitivity side of spacecraft C4, in Particle_Energy_Flux units",5640,2020-12-31T23:59:59.000Z,CIS,cef,O+ 3D distribution (High sensitivity),2011-11-29T16:36:29.375Z,CIS-CODIF4,,1,True,True,0,Calibrated,Science,2020-12-31T00:00:00.000Z,4.0,"Ion_Composition, Thermal_Plasma, Energetic_Particles",16.0,0,0,Cluster-4,0,Iannis Dandouras>PI>Iannis.Dandouras@irap.omp.eu ---- Henri Reme>Deputy-PI>Henri.Reme@irap.omp.eu,0,0,1,2001-01-01T00:00:00.000Z,4.0,"CIS-CODIF 3D Oxygen+ distributions on C4, High-Sensitivity, in Particle_Energy_Flux",Particle Energy Flux
Please acknowledge the ESA Cluster Active Archive in any publication based upon use of this data,0,General,1,0,See User Guide for caveats,C4_CG_AUX_CAA_6HR_PARTICLES1_PS,,,CAA_Graphic,CG,2021-08-03T06:17:06.672Z,2021-03-05T03:21:48.631Z,"This plot, available for each spacecraft, is based on the calibrated CAA data and has a similar layout as the corresponding CSDSweb plot (that is based on default calibration values). The panels from top to bottom are: Total magnetic field (FGM) Total ion velocity (CIS-HIA) Polar angle of Ion velocity (CIS-HIA) Azimuthal angle of ion velocity (CIS-HIA) Total ion density (CIS-HIA) Ion energy spectra (CIS) Electron energy spectra (PEACE)",30160,2020-10-01T00:00:00.000Z,AUX,ps,6-hr CAA Summary Plot (Particles1),2015-03-03T16:41:46.471Z,AUX4,,0,False,False,0,Calibrated,Graphical,2020-09-30T18:00:00.000Z,21600.0,Status,21600.0,0,0,Cluster-4,0,caaops>Non-PI>caaops@sciops.esa.int,0,0,1,2001-01-01T00:00:00.000Z,21600.0,6-hr CAA Summary Plot (Particles1),Postscript
Please acknowledge the PEACE instrument team and ESA Cluster Active Archive in any publication based upon use of this data.,0,Particle Distribution,1,0,*C1_CQ_P

### Using WHERE to restrict results

To get a list of all datasets that include FGM, we need to query the v_dataset table for the dataset_id and add the WHERE statement to restrict the results to those that contain 'FGM'. Remember that the percentage sign (%) is the wildcard, but it doesn't need to be URL encoded here.

In [22]:
results = CSA.launch_job("SELECT dataset_id FROM csa.v_dataset WHERE dataset_id like '%FGM%'")
r = results.get_results()
r

dataset_id
object
C3_CG_MULT_COMP_E_HIA_FGM_EFW_TS_PS
C3_CG_MULT_COMP_E_HIA_FGM_EFW_TS_PNG
C1_CG_MULT_COMP_E_HIA_FGM_EFW_PS
C2_CG_MULT_COMP_E_HIA_FGM_EFW_TS_PNG
C2_CG_MULT_COMP_E_HIA_FGM_EFW_TS_PS
C4_CG_MULT_COMP_E_HIA_FGM_EFW_TS_PS
C1_CG_MULT_COMP_E_HIA_FGM_EFW_TS_PS
C4_CP_FGM_SPIN_ISR2
C1_CG_MULT_COMP_E_HIA_FGM_EFW_TS_PNG


### An inventory file request:

Request (SELECT) 5 columns FROM csa.v_dataset_inventory table WHERE the dataset_id is 'C1_CP_FGM_SPIN' AND start_time is *before* 1 May 2002 AND end_time is *after* 1 April 2002 and ORDER BY the start time column, and print the results, as text or as an [astropy.table](https://docs.astropy.org/en/stable/table/index.html?highlight=astropy.table):

In [4]:
results = CSA.launch_job("SELECT dataset_id,start_time,end_time,num_instances,inventory_version "
                         "FROM csa.v_dataset_inventory "
                         "WHERE dataset_id='C1_CP_FGM_SPIN' "
                         "AND start_time<='2002-05-01T00:00:00' " 
                         "AND end_time>='2002-04-01T00:00:00' "
                         "ORDER BY start_time")

# Either print the results as text, or as an astropy.table (which can be easier to read)
#print(results.get_results())

r = results.get_results()
r

dataset_id,start_time,end_time,num_instances,inventory_version
object,object,object,int32,int32
C1_CP_FGM_SPIN,2002-03-30T18:17:57.879Z,2002-04-01T00:00:00.000Z,0,10
C1_CP_FGM_SPIN,2002-04-01T00:00:00.000Z,2002-04-01T00:21:04.972Z,0,10
C1_CP_FGM_SPIN,2002-04-01T00:21:04.972Z,2002-04-01T00:28:51.002Z,117,10
C1_CP_FGM_SPIN,2002-04-01T00:28:51.002Z,2002-04-01T00:35:16.683Z,0,10
C1_CP_FGM_SPIN,2002-04-01T00:35:16.683Z,2002-04-01T03:36:48.128Z,2704,10
C1_CP_FGM_SPIN,2002-04-01T03:36:48.128Z,2002-04-01T18:41:24.702Z,0,10
C1_CP_FGM_SPIN,2002-04-01T18:41:24.702Z,2002-04-01T18:48:50.646Z,112,10
C1_CP_FGM_SPIN,2002-04-01T18:48:50.646Z,2002-04-01T18:55:20.346Z,0,10
C1_CP_FGM_SPIN,2002-04-01T18:55:20.346Z,2002-04-02T06:44:49.979Z,10534,10
